# DGDCN EEG seizure detection: project progress

This notebook records the 200 Hz reproduction attempt and provides Python helpers for reading sanitized run state and aggregate metrics. It does not download or open EEG data. The TUSZ corpus is restricted research data and is not stored in this repository.

The starting paper is Zhang, Dai, and Guo, *Dynamic Graph Convolutional Network with Dilated Convolution for Epilepsy Seizure Detection* (2025), [doi:10.3390/bioengineering12080832](https://doi.org/10.3390/bioengineering12080832). This experiment adapts the public DGDCN code to available TUSZ v2.0.6 feature archives. It is not an exact reproduction.

## Experiment setup

| Item | Current choice |
|---|---|
| Task | Binary classification of 12-second EEG clips |
| Corpus | TUSZ v2.0.6, restricted access |
| Feature rate | 200 Hz |
| Representation | Log-amplitude FFT, 100 bins |
| Electrode nodes | 19, following the released model code |
| Partitions | Official patient-separated train, development, and evaluation sets |
| Training pool | All marked seizure clips plus a seeded equal-count sample of non-seizure clips |
| Patient sampling | Replacement weights give each training patient's clips equal total sampling mass; per-epoch class balance is not guaranteed |
| Accelerator checked | Colab NVIDIA Tesla T4; synthetic forward/backward check passed |

The paper describes TUSZ v2.0.1 and 22 derivations. This project uses v2.0.6 and the 19-node representation in the released code. The 256 Hz path was discussed but has not been run.

## Historical chronology through the initial staging restart

1. A critical literature review established the clinical and evaluation context for EEG seizure detection. DGDCN was selected as a concrete reproduction case study.
2. Inspection of the public implementation found execution and configuration problems, including hard-coded machine paths and differences between the paper's channel description and the code. The initial target was the 12-second configuration.
3. The authorized corpus was processed into 40 200 Hz feature archive batches with corresponding annotation-marker archives. The outlier report was set aside; its suggested exclusions were not applied.
4. A private Kaggle run passed a synthetic CUDA forward/backward check. Staging stopped after three feature batches because the shared-file download quota was reached. Normalization and training did not start. A later private dataset upload was accepted, but Kaggle's dataset APIs returned HTTP 403, so availability could not be confirmed.
5. Colab later provided a Tesla T4. Drive access and an initial Drive checkpoint write were verified before the cached archives were restored.
6. On 2026-10-09 at 12:22 UTC, 36 of 40 batches had verified staging checkpoints. The next Drive transfer ended with an incomplete response. The controller stopped safely. Normalization, model training, and evaluation had not begun. Completed staging checkpoints and a failure log were saved to Drive.

That is a snapshot of one run. Add later retries as separate dated entries in the project log.

A resumed attempt began at 12:33 UTC on 2026-10-09. It reuses verified staging checkpoints, restored 402,653,184 bytes from the interrupted batch 37 archive, and writes outputs to a new Drive run folder. At the latest recorded log, it was still staging; normalization and training had not started.

## Historical snapshots

The next cell records only progress facts. It does not contain private dataset or Drive identifiers.

In [ ]:
last_verified_state = {
    'observed_utc': '2026-10-09T12:22:16Z',
    'accelerator': 'NVIDIA Tesla T4',
    'phase': 'staging',
    'completed_batches': 36,
    'total_batches': 40,
    'interrupted_batch': 37,
    'error_type': 'IncompleteRead during Drive archive transfer',
    'drive_checkpoint_write_probe': 'passed',
    'normalization_started': False,
    'training_started': False,
    'evaluation_started': False,
}
print(last_verified_state)

## Training snapshot (2026-10-09 13:03:46 UTC)

The 40/40 feature batches were restored and verified, training-only normalization completed, and training was active on a Tesla T4. Two development epochs had completed. This snapshot is historical and should not be mistaken for the latest live run state. The held-out evaluation had not run. These are clip-level metrics, not event-level detection metrics.

In [ ]:
latest_verified_training = {
    'observed_utc': '2026-10-09T13:03:46Z',
    'accelerator': 'NVIDIA Tesla T4',
    'verified_feature_batches': '40/40',
    'feature_files': 374112,
    'feature_bytes': 34880512545,
    'marker_rows': 373792,
    'training_normalization_complete': True,
    'training_active': True,
    'evaluation_started': False,
    'dev_clips': 73387,
    'dev_positive_clips': 2310,
    'dev_negative_clips': 71077,
    'all_negative_accuracy_baseline': 71077 / 73387,
    'epoch_1': {
        'loss': 0.3663629473, 'accuracy': 0.8929646940, 'f1': 0.1624906706,
        'precision': 0.1077945961, 'recall': 0.3298701299,
        'specificity': 0.9112652475, 'auroc': 0.7476976831,
        'average_precision': 0.0774383501, 'threshold': 0.5005499125,
    },
    'epoch_2': {
        'loss': 0.2625114316, 'accuracy': 0.9034297628, 'f1': 0.1719827083,
        'precision': 0.1177788446, 'recall': 0.3186147186,
        'specificity': 0.9224362311, 'auroc': 0.7604461371,
        'average_precision': 0.0962714109, 'threshold': 0.4524217248,
    },
    'caveat': 'Two development epochs from one run; no CI; no held-out or event-level result.',
}
print(latest_verified_training)


## Training snapshot (2026-10-09 13:13:42 UTC)

The 40 feature batches are staged, training-only normalization is complete, and the T4 training controller and Drive sync process are alive. Four epochs have completed. Epoch 4 is the best development-AUROC checkpoint so far. Development metrics are clip-level, from one run, and are not held-out or clinical event-detection results; final evaluation has not started.

In [ ]:
latest_verified_training = {
    'observed_utc': '2026-10-09T13:13:42Z',
    'accelerator': 'NVIDIA Tesla T4',
    'verified_feature_batches': '40/40',
    'training_normalization_complete': True,
    'training_active': True,
    'drive_sync_active': True,
    'drive_sync_error': None,
    'completed_epochs': 4,
    'best_epoch_so_far': 4,
    'epoch_3_dev': {'auroc': 0.7671610363, 'average_precision': 0.1135137281},
    'epoch_4_dev': {
        'loss': 0.3727751168, 'accuracy': 0.9340346383, 'f1': 0.1878879383,
        'precision': 0.1533826349, 'recall': 0.2424242424,
        'specificity': 0.9565119518, 'auroc': 0.7694211393,
        'average_precision': 0.1238358195, 'threshold': 0.70188242197,
    },
    'paper_reported_12s_auc': 0.887,
    'held_out_evaluation_started': False,
    'caveat': 'Development clip metrics; one run; no CI; training remains active.',
}
print(latest_verified_training)


## Optional: summarize a later run-state file

Set RUN_STATE_PATH to a sanitized latest-state.json file to inspect a later attempt. The helper prints only an allowlist; it does not dump arbitrary state that may contain private paths or identifiers.

In [ ]:
import json
from pathlib import Path

RUN_STATE_PATH = Path('latest-state.json')  # Point to a sanitized run-state copy.

def summarize_state(path):
    path = Path(path)
    if not path.is_file():
        return {'state_file_found': False}
    raw = json.loads(path.read_text(encoding='utf-8'))
    allowed = ('status', 'phase', 'gpu', 'completed_batches', 'current_batch',
               'downloaded_archives', 'normalization_started',
               'normalization_complete', 'trainer_process_started',
               'training_started', 'elapsed_seconds', 'updated_utc')
    result = {key: raw[key] for key in allowed if key in raw}
    result['state_file_found'] = True
    if 'error' in raw:
        result['error_type'] = str(raw['error']).split(':', 1)[0][:80]
    return result

summarize_state(RUN_STATE_PATH)

## Optional: summarize aggregate clip metrics

Point METRICS_PATH to aggregate metrics.jsonl after training begins. The helper reads only aggregate fields. It does not open prediction rows or participant-level records. A missing file means no metrics are available in this notebook; it does not mean a score of zero.

In [ ]:
METRICS_PATH = Path('metrics.jsonl')  # Point to aggregate epoch metrics, if available.
METRIC_KEYS = ('loss', 'accuracy', 'sensitivity', 'specificity', 'f1',
               'auroc', 'average_precision', 'threshold')

def read_aggregate_metrics(path):
    path = Path(path)
    if not path.is_file():
        return {'metrics_file_found': False, 'epochs_available': 0}
    rows = []
    for line in path.read_text(encoding='utf-8').splitlines():
        if not line.strip():
            continue
        row = json.loads(line)
        dev = row.get('dev', {})
        rows.append({'epoch': row.get('epoch'),
                     'train_loss': row.get('train_loss'),
                     'dev': {key: dev[key] for key in METRIC_KEYS if key in dev}})
    return {'metrics_file_found': True, 'epochs_available': len(rows),
            'latest_epoch': rows[-1] if rows else None}

read_aggregate_metrics(METRICS_PATH)

## Reading a future result

This run is clip-level classification. Report held-out clip metrics with the split, patient count, threshold-selection rule, and dataset and code versions. Do not describe clip AUROC or accuracy as event sensitivity, false alarms per 24 hours, or detection latency. The present pipeline alone does not establish clinical utility.

## References

- Zhang X, Dai C, Guo Y. Dynamic Graph Convolutional Network with Dilated Convolution for Epilepsy Seizure Detection. *Bioengineering*. 2025;12(8):832. [DOI](https://doi.org/10.3390/bioengineering12080832).
- Inspected public DGDCN source snapshot: commit `96c7bee49c240124735ffc2ab2ad118d27a1ff97`.
- [Project progress log](../docs/PROJECT_PROGRESS.md).

### 10. Runtime replacement and stale monitor output

A fresh diagnostic at **2026-10-09 14:23:42 UTC** found a runtime with about 267 seconds of uptime, no `nvidia-smi`, only `.config` and `sample_data` in `/content`, and no training output directory or training processes. The earlier monitor returned saved notebook output; its process-alive flags did not describe this new runtime. The earlier suggestion of an epoch-4 stall was therefore unsupported.

Direct Drive reads showed **seven completed epochs**. The saved metrics file was modified at 13:33:59 UTC, `last.pt` at 13:33:57 UTC, and the controller log at 13:34:02 UTC. Epochs 5, 6, and 7 had development AUROC 0.7591, 0.7671, and 0.7675, respectively. Epoch 4 remains best by development AUROC (0.7694). The `best.pt` file remains present, last modified at 13:13:39 UTC. The checkpoint files' internal contents have not yet been loaded after this interruption. The saved run folder contains no final evaluation artifact.

The available evidence establishes runtime replacement and stale monitoring output. It does not identify whether the previous session ended because of a user action, quota, disconnection policy, or another cause. No new training exception appears after the final saved epoch. The DataLoader worker warning is not sufficient evidence of a deadlock.

Recovery must explicitly load the saved training checkpoint; the controller's existing launch command does not automatically pass the trainer's supported `--resume` argument. Preserve both the latest checkpoint and the best development checkpoint. Restore the existing features and saved training normalization. Future monitoring must include a fresh execution timestamp and invocation marker, and reject old notebook outputs as live evidence.

### 12. CPU checkpoint recovery

At **2026-10-10 05:33:55 UTC**, the saved checkpoint was reopened and verified with seven completed epochs, epoch 4 still the best checkpoint, and seven metric rows available. The CPU continuation controller and checkpoint synchronizer started in a new output run folder. Continuation is set to begin at epoch 8. CPU continuation is an authorized option when GPU runtime is unavailable. The checkpoint preserves training normalization, optimizer, scheduler, Python/NumPy/CPU Torch RNG, and early-stopping state. CUDA RNG does not drive CPU dropout, and hardware continuation is not bitwise identical.

At the 2026-10-10 05:37:14 UTC check, the controller and synchronizer were alive with no reported error. Batches **1-3 of 40** were complete, and batch 4 was transferring. The controller is restaging the **40 saved feature and marker archive pairs**, totaling **30,071,690,466 compressed bytes**. This restores existing 200 Hz transformed features and does not resample EDF data. Training had not started, and evaluation had not started.


## Implementation fidelity review

The [implementation audit](../docs/IMPLEMENTATION_AUDIT.md) documents the full-paper and code review. The executed core architecture and max-pooled binary head were retained. Normalization and patient sampling changed, and the marker populations differ substantially. These are candidate explanations requiring controlled comparisons; none has yet been shown to cause the AUROC gap. No new training was launched.


## Completed run summary (verified 2026-10-10 08:16 UTC)

Eight training epochs completed and the run stopped by its configured early-stopping rule. The best checkpoint was epoch 4 by development AUROC. One held-out evaluation completed at threshold 0.701882, selected to maximize development F1. Eval AUROC was 0.840075, clip recall 0.286029, and specificity 0.968545. The 38,384 clips included 2,269 positive and 36,115 negative clips; confusion counts were TP 649, FN 1,620, FP 1,136, and TN 34,979. These are clip counts, not event counts.

The subsequent sensitivity analysis selected thresholds from development-positive scores at recall targets 0.85, 0.90, 0.95, and 0.99 before reading evaluation scores. The primary dev-0.95 threshold was 0.07521818578243256. Evaluation clip recall rose to 97.58%, with 55 false-negative clips and 28,014 false-positive clips (77.57% of background clips). AUROC remained 0.840075. See the [complete threshold table](../docs/SENSITIVITY_DIAGNOSIS.md) and [primary operating-point configuration](../results/primary_operating_point.json).

Patient-equal sampling remained in place. Its expected positive draw fraction was 16.78% despite the original balanced clip pool; the effect on score quality requires a controlled training comparison. No event sensitivity, missed-event count, false alarms per hour, or latency has been measured.

See the [sensitivity diagnosis](../docs/SENSITIVITY_DIAGNOSIS.md), [sanitized aggregate evaluation](../results/final_eval_metrics.json), and [project chronology](../docs/PROJECT_PROGRESS.md#13-completed-run-and-sensitivity-diagnosis).

## Verified data quality review (2026-10-10)

A read-only audit checked 8,140 TUSZ v2.0.6 EDF headers and payload layouts. It excluded 625 files that lack a required model electrode, four recordings with a required EEG signal proven constant across every sample, and one byte-identical duplicate. The 625 montage-ineligible files were already absent from the cached feature markers. All 134 distinct flat or dead-channel candidates from the prior reports were matched and checked across their full required-channel samples.

The saved marker tables were reconciled to the source QC manifest. The filtered view has 373,636 clips, 156 fewer than the old 373,792. Every removed cached clip is a training negative (155 from the four constant-channel recordings and one duplicate). Development and evaluation rows are unchanged. Official patient partitions were retained. The six loader-ready marker tables and a sanitized report were uploaded to the existing collaborator training folder in Drive; sharing settings were unchanged.

The official NEDC annotation guide says a short bckg entry marks seizure-free files so their annotation files are non-empty. The audit preserves the established one-second seizure-overlap threshold and treats annotation coverage gaps as review flags. It omits one zero-duration row in the derived annotation overlay; the source annotation remains unchanged.

The filtered balanced training pool contains 11,223 positive and 11,223 negative clips across 575 patients. Under the existing patient-balanced sampler, the expected positive draw probability is 0.167957; 4.9539 is a candidate positive loss weight for an ablation only. The prior run used unweighted binary cross-entropy. No training or feature rewriting was done in this audit. The next model run must use the filtered training markers and recompute normalization from retained training features. The data filter does not itself establish an AUROC improvement. See [the full data quality review](../docs/DATA_QUALITY_REVIEW.md).